In [1]:
import pandas as pd
import getpass
from sqlalchemy import create_engine, text, URL

print("Todo listo")

Todo listo


In [2]:
password = getpass.getpass("Contraseña de MySQL: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database="sakila"
)

engine = create_engine(connection_url)

In [3]:
with engine.connect() as connection:
    query = text("SELECT DATABASE()")
    result = connection.execute(query)
    print("Conectado a:", result.scalar())

Conectado a: sakila


In [4]:
def rentals_month(engine, month, year):
    query = text("""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year
    """)

    with engine.connect() as connection:
        result = connection.execute(
            query,
            {"month": month, "year": year}
        )

        df = pd.DataFrame(result.fetchall(), columns=result.keys())

    return df

In [5]:
rentals_may = rentals_month(engine, 5, 2005)
rentals_june = rentals_month(engine, 6, 2005)

display(rentals_may.head())
display(rentals_june.head())

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


In [6]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    counts = (
        df.groupby("customer_id")["rental_id"]
        .count()
        .reset_index(name=column_name)
    )

    return counts

In [7]:
counts_may = rental_count_month(rentals_may, 5, 2005)
counts_june = rental_count_month(rentals_june, 6, 2005)

display(counts_may.head())
display(counts_june.head())

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [8]:
def compare_rentals(df1, df2):
    column1 = df1.columns[1]
    column2 = df2.columns[1]

    comparison = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="inner"
    )

    comparison["difference"] = (
        comparison[column2] - comparison[column1]
    )

    return comparison

In [9]:
comparison = compare_rentals(counts_may, counts_june)

display(comparison.head())
print("Clientes activos en ambos meses:", len(comparison))

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


Clientes activos en ambos meses: 512
